In [6]:
import pandas as pd
import networkx as nx
import os
from dotenv import load_dotenv, find_dotenv
import igraph as ig

In [7]:
load_dotenv(find_dotenv())
root = os.getenv("ROOT_DIR")

graph_df = pd.read_csv(os.path.join(root, 'data', 'cleaned_collaboration.csv'))

In [8]:
print(graph_df.shape)
print(graph_df.columns)

(188305, 14)
Index(['id_1', 'id_2', 'name_1', 'followers_1', 'popularity_1', 'genres_1',
       'chart_hits_1', 'top_chart_1', 'name_2', 'followers_2', 'popularity_2',
       'genres_2', 'chart_hits_2', 'top_chart_2'],
      dtype='object')


In [9]:
import math
from pathlib import Path
from typing import Literal, Tuple

import igraph as ig
import pandas as pd
from tqdm import tqdm


def build_graph(
    df: pd.DataFrame,
    root: str | Path = ".",
    layout: Literal["openord", "fr"] = "openord",      # choose: "openord" or "fr"
    community_algo: Literal["multilevel", "louvain"] = "multilevel",
) -> Tuple[ig.Graph, pd.DataFrame, pd.DataFrame]:
    """
    Build an undirected igraph from a two‑column edge DataFrame, compute a 2‑D
    layout (OpenOrd by default, Fruchterman–Reingold if layout="fr"), detect
    communities, and return:

        g          – igraph.Graph with all vertex attributes
        nodes_df   – DataFrame: id,x,y,z,degree,community + original attrs
        edges_df   – DataFrame: source,target  (IDs, not indices)

    Parameters
    ----------
    df : pd.DataFrame
        Original edge list plus duplicated node‑side attributes, e.g.
        id_1, id_2, name_1, name_2, …
    root : str | Path
        Unused by default – left in for API compatibility.
    layout : {"openord", "fr"}
        Which layout algorithm to use on the giant component.
    community_algo : {"multilevel", "louvain"}
        Which igraph community detection method to run.
    """

    # 1) collect unique node attributes ------------------------------------
    node_attrs, dropped_idx = {}, set()
    for side in (1, 2):
        id_col = f"id_{side}"
        cols = {k: f"{k}_{side}" for k in (
            "name", "followers", "popularity",
            "genres", "chart_hits", "top_chart")
        }

        for idx, row in tqdm(df.iterrows(), total=len(df),
                             desc=f"Collect attrs side {side}", unit="row"):
            if idx in dropped_idx:
                continue

            nid = row[id_col]
            attrs = {k: row[v] for k, v in cols.items()}

            if nid in node_attrs:
                # keep the first non‑conflicting copy
                for k, v in attrs.items():
                    if k in ("chart_hits", "top_chart"):
                        continue
                    if node_attrs[nid][k] != v:
                        dropped_idx.add(idx)
                        break
            else:
                node_attrs[nid] = attrs

    # 2) build edge list (skip conflicting rows) ---------------------------
    ids = list(node_attrs)
    id2idx = {nid: i for i, nid in enumerate(ids)}
    edges = []
    for i, row in tqdm(df.iterrows(), total=len(df),
                       desc="Building edges", unit="row"):
        if i in dropped_idx:
            continue
        edges.append((id2idx[row.id_1], id2idx[row.id_2]))

    # 3) create igraph -----------------------------------------------------
    g = ig.Graph(n=len(ids), edges=edges, directed=False)
    for attr in ("name", "followers", "popularity",
                 "genres", "chart_hits", "top_chart"):
        g.vs[attr] = [node_attrs[n][attr] for n in ids]
    g.vs["degree"] = g.degree()

    # 4) community detection ----------------------------------------------
    if community_algo == "multilevel":
        comms = g.community_multilevel()
    elif community_algo == "louvain":
        comms = g.community_louvain()
    else:
        raise ValueError(f"Unknown community algorithm '{community_algo}'")
    membership = comms.membership
    g.vs["community"] = membership

    # 5) compute layout ----------------------------------------------------
    main_ids = [v.index for v in g.vs if v["degree"] > 0]
    iso_ids = [v.index for v in g.vs if v["degree"] == 0]
    coords = [[math.nan, math.nan] for _ in range(g.vcount())]

    if main_ids:
        sub = g.induced_subgraph(main_ids)
        try:
            if layout == "openord":
                lay = sub.layout("openord", dim=2)
            elif layout == "fr":
                lay = sub.layout("fr")
            else:
                raise ValueError(f"Unknown layout '{layout}'")
        except (KeyError, ig.InternalError):
            lay = sub.layout("lgl")   # fallback
        for loc, glob in enumerate(main_ids):
            coords[glob] = lay[loc]

    if iso_ids:
        r = max(max(abs(x), abs(y)) for x, y in coords if not math.isnan(x)) or 1.0
        for k, vid in tqdm(enumerate(iso_ids), total=len(iso_ids),
                           desc="Positioning isolates", unit="iso"):
            phi = 2 * math.pi * k / len(iso_ids)
            coords[vid] = (r * 1.05 * math.cos(phi),
                           r * 1.05 * math.sin(phi))

    # 6) normalise x, y and compute z --------------------------------------
    xs = [c[0] for c in coords]
    ys = [c[1] for c in coords]
    x_min, x_max = min(xs), max(xs)
    y_min, y_max = min(ys), max(ys)

    def norm(v, vmin, vmax) -> float:
        return 5000 * (v - vmin) / (vmax - vmin) if not math.isnan(v) else v

    xs_norm = [norm(x, x_min, x_max) for x in xs]
    ys_norm = [norm(y, y_min, y_max) for y in ys]
    degrees = g.vs["degree"]
    zs = [deg * 2 for deg in degrees]

    # 7) assemble DataFrames ----------------------------------------------
    nodes_df = (
        pd.DataFrame.from_dict(node_attrs, orient="index")
          .reset_index().rename(columns={"index": "id"})
          .assign(x=xs_norm, y=ys_norm, z=zs,
                  degree=degrees, community=membership)
    )

    edges_df = (
        pd.DataFrame(edges, columns=["src_idx", "tgt_idx"])
          .assign(source=lambda d: d.src_idx.map(ids.__getitem__),
                  target=lambda d: d.tgt_idx.map(ids.__getitem__))
          .loc[:, ["source", "target"]]
    )

    return g, nodes_df, edges_df


In [10]:
g, nodes, edges = build_graph(graph_df)



Building edges: 100%|██████████| 188305/188305 [00:05<00:00, 35088.68row/s]
c:\Python312\Lib\site-packages\igraph\layout.py:691: RuntimeWarning: LGL layout does not support disconnected graphs yet. at src/layout/large_graph.c:179
  layout = func(*args, **kwds)


In [11]:
# g.to_csv(os.path.join(root, 'data', 'constructed_network.csv'))

In [12]:
nodes.to_csv(os.path.join(root, 'data', 'network_nodes.csv'))
edges.to_csv(os.path.join(root, 'data', 'network_edges.csv'))